# CONCODE Brain — Kaggle GPU API (temporary prototype)

Runs the existing ConCode FastAPI travel endpoint and your real CodeLlama GGUF from an attached Kaggle input dataset. It does not run on your computer or replace ConCode with another model. A Kaggle GPU session can stop or be reclaimed, and the free Cloudflare quick-tunnel URL changes on restart, so this is temporary—not always-on hosting.

## Before running

1. Sign in to Kaggle and create a **private Dataset** containing the supplied `concode_kaggle_dataset.zip` archive. It includes only the required ConCode source/configuration/model files; it excludes `.env`, `venv/`, API keys, and stored profiles. Respect the model license. The archive is several GB.
2. Create a Kaggle Notebook, attach the private dataset, enable **GPU** and **Internet** in notebook settings, then run the cells in order. The notebook finds the attached dataset containing `src/service.py` automatically.
3. In the notebook's **Add-ons → Secrets**, add `CONCODE_API_KEY` with a private random value at least 32 characters long. Enable notebook access to that secret. The key is never printed or saved in the notebook output.

The API binds to loopback inside the Kaggle VM. Cloudflare creates a temporary HTTPS tunnel only after the real model is loaded and an authenticated itinerary request succeeds. Protected routes use `X-API-Key`. Do not put the key in browser code or share the public URL without it; stop the notebook when finished.

In [ ]:
from pathlib import Path
import zipfile

INPUT_ROOT = Path('/kaggle/input')
archives = sorted(INPUT_ROOT.glob('*/concode_kaggle_dataset.zip'))
if archives:
    if len(archives) != 1:
        raise RuntimeError(f'Attach exactly one ConCode dataset archive; found {len(archives)}.')
    PROJECT_ROOT = Path('/kaggle/working/CONCODE_BRAIN')
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        root = PROJECT_ROOT.resolve()
        for member in archive.infolist():
            target = (PROJECT_ROOT / member.filename).resolve()
            if not target.is_relative_to(root):
                raise RuntimeError('Dataset archive contains an unsafe path.')
        archive.extractall(PROJECT_ROOT)
else:
    candidates = sorted(INPUT_ROOT.glob('*/src/service.py'))
    if len(candidates) != 1:
        raise RuntimeError(
            'Attach the private ConCode dataset archive or a dataset folder containing src/service.py. '
            f'Found {len(candidates)} source folders.'
        )
    PROJECT_ROOT = candidates[0].parent.parent
if not (PROJECT_ROOT / 'src/service.py').is_file():
    raise RuntimeError('ConCode service source is missing.')
required = [
    PROJECT_ROOT / 'data/guilds.yaml',
    PROJECT_ROOT / 'models/codellama-7b-concode.Q4_K_M.gguf',
    PROJECT_ROOT / 'models/intent_classifier/config.json',
    PROJECT_ROOT / 'models/code_repair/config.json',
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError('The attached dataset is missing required ConCode files: ' + ', '.join(missing))
model_size_gb = required[1].stat().st_size / 1_000_000_000
print('ConCode project located in attached Kaggle input.')
print('GGUF model size (GB):', round(model_size_gb, 2))
print('Input is read-only; runtime profile data and logs will be stored in /kaggle/working.')


In [ ]:
import os
import shutil
import subprocess
import sys
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU is active. Enable GPU in Kaggle Notebook options and rerun.')
if not shutil.which('nvcc'):
    raise RuntimeError('nvcc is unavailable. This runtime cannot build CUDA llama.cpp; stopping instead of silently running inference on CPU.')
print('GPU:', torch.cuda.get_device_name(0), '| CUDA:', torch.version.cuda)

dependencies = [
    'fastapi>=0.100.0', 'uvicorn[standard]>=0.22.0', 'transformers>=4.30.0',
    'sentence-transformers>=2.2.2', 'faiss-cpu>=1.7.4', 'pydantic>=2.0',
    'pyyaml>=6.0', 'python-dotenv>=1.0.0', 'requests>=2.31.0', 'cmake', 'ninja', 'scikit-build-core'
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *dependencies])
build_env = os.environ.copy()
build_env['CMAKE_ARGS'] = '-DGGML_CUDA=on'
build_env['FORCE_CMAKE'] = '1'
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '--quiet', '--no-cache-dir', 'llama-cpp-python'
], env=build_env)

from llama_cpp import llama_cpp
if not llama_cpp.llama_supports_gpu_offload():
    raise RuntimeError('llama.cpp was not built with CUDA GPU offload; refusing to launch CPU-only inference.')
print('CUDA-enabled llama.cpp verified.')


In [ ]:
import time
import requests
from kaggle_secrets import UserSecretsClient

try:
    API_KEY = UserSecretsClient().get_secret('CONCODE_API_KEY')
except Exception as exc:
    raise RuntimeError('Add CONCODE_API_KEY under Kaggle Add-ons → Secrets and enable notebook access.') from exc
if not API_KEY or len(API_KEY) < 32:
    raise RuntimeError('CONCODE_API_KEY must contain at least 32 characters.')

os.environ['CONCODE_API_KEY'] = API_KEY
os.environ['CONCODE_REQUIRE_API_KEY'] = '1'
os.environ['CONCODE_INFERENCE_BACKEND'] = 'local'
os.environ['CONCODE_MODEL_DIR'] = str(PROJECT_ROOT / 'models')
os.environ['CONCODE_USER_PROFILES_PATH'] = '/kaggle/working/concode_user_profiles.json'
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'

log_file = open('/kaggle/working/concode-uvicorn.log', 'w')
server = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'src.service:app', '--host', '127.0.0.1', '--port', '8000', '--workers', '1'],
    cwd=PROJECT_ROOT,
    env=os.environ.copy(),
    stdout=log_file,
    stderr=subprocess.STDOUT,
)
BASE_URL = 'http://127.0.0.1:8000'
for _ in range(180):
    if server.poll() is not None:
        raise RuntimeError('ConCode exited during model startup. Inspect /kaggle/working/concode-uvicorn.log.')
    try:
        if requests.get(BASE_URL + '/ready', timeout=3).ok:
            break
    except requests.RequestException:
        pass
    time.sleep(5)
else:
    raise TimeoutError('ConCode did not become ready within 15 minutes. Inspect /kaggle/working/concode-uvicorn.log.')

headers = {'X-API-Key': API_KEY}
status_response = requests.get(BASE_URL + '/api/v1/status', headers=headers, timeout=15)
if not status_response.ok:
    raise RuntimeError('Authenticated ConCode status check failed: HTTP ' + str(status_response.status_code))
status = status_response.json()
generator = status.get('subsystems', {}).get('code_generator', {})
if not generator.get('loaded') or generator.get('backend') != 'local':
    raise RuntimeError('The real local GGUF model is not loaded; no public tunnel will be opened.')
print('Authenticated ConCode API ready; GGUF backend loaded.')


In [ ]:
# Test the exact TripVerse travel endpoint before exposing any public URL.
test_response = requests.post(
    BASE_URL + '/api/v1/travel',
    headers={**headers, 'Content-Type': 'application/json'},
    json={
        'query': 'Create a one-day relaxed itinerary for Goa with two travelers.',
        'trip_context': {
            'destination': 'Goa', 'duration_days': 1, 'travelers_count': 2,
            'currency': 'INR', 'preferences': ['relaxed', 'local food']
        },
        'mode': 'itinerary'
    },
    timeout=300,
)
if not test_response.ok:
    raise RuntimeError('Real ConCode itinerary inference failed with HTTP ' + str(test_response.status_code) + '. No tunnel will be opened.')
test_result = test_response.json()
try:
    itinerary = __import__('json').loads(test_result['reply'])
except Exception as exc:
    raise RuntimeError('ConCode response was not valid itinerary JSON. No tunnel will be opened.') from exc
if test_result.get('model_backend') != 'local' or len(itinerary.get('itinerary', [])) != 1:
    raise RuntimeError('Travel response did not pass the real-model/day-count check. No tunnel will be opened.')
print('Real authenticated /api/v1/travel inference succeeded.')
print('Model backend:', test_result['model_backend'], '| days:', len(itinerary['itinerary']))
print('Estimated plan output is not printed in notebook logs.')


In [ ]:
# Cloudflare quick tunnels need no account token, but their public hostname is temporary.
cloudflared_path = '/kaggle/working/cloudflared'
download = subprocess.run(
    ['wget', '-q', '-O', cloudflared_path, 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'],
    check=False,
)
if download.returncode != 0 or not Path(cloudflared_path).is_file():
    raise RuntimeError('Could not download cloudflared. Check Kaggle Notebook Internet settings.')
os.chmod(cloudflared_path, 0o755)
tunnel_log = open('/kaggle/working/cloudflared.log', 'w')
tunnel = subprocess.Popen(
    [cloudflared_path, 'tunnel', '--no-autoupdate', '--url', BASE_URL],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
)
import re
PUBLIC_URL = None
for _ in range(60):
    if tunnel.poll() is not None:
        raise RuntimeError('Cloudflare tunnel exited; inspect /kaggle/working/cloudflared.log.')
    try:
        tunnel_output = Path('/kaggle/working/cloudflared.log').read_text(errors='replace')
    except OSError:
        tunnel_output = ''
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', tunnel_output)
    if match:
        PUBLIC_URL = match.group(0)
        break
    time.sleep(2)
if not PUBLIC_URL:
    raise TimeoutError('Cloudflare did not provide an HTTPS URL. Inspect /kaggle/working/cloudflared.log.')

# Confirm the public tunnel reaches this running ConCode process.
public_health = requests.get(PUBLIC_URL + '/health', timeout=30)
if not public_health.ok:
    raise RuntimeError('Public tunnel health check failed; do not connect TripVerse.')
print('Temporary HTTPS ConCode URL:', PUBLIC_URL)
print('TripVerse API: POST ' + PUBLIC_URL + '/api/v1/travel')
print('Protected routes require X-API-Key. Keep the key only in the Vercel server-side secret store.')
print('This URL works only while this Kaggle session and tunnel stay active. Do not treat it as always-on hosting.')


## Connect TripVerse only after the public endpoint is verified

Keep the notebook running. In the TripVerse Vercel project, add Production secrets:

- `CONCODE_API_URL` = the printed `https://...trycloudflare.com` base URL (no `/api/v1/travel` suffix).
- `CONCODE_API_KEY` = the same private value used for the Kaggle Secret.

Then redeploy TripVerse and test a planner request. Never place `CONCODE_API_KEY` in notebook code, browser JavaScript, or a public repository. Kaggle GPU quotas, internet availability, inactivity shutdown, and Cloudflare quick-tunnel availability can change; the hostname changes after restarting the tunnel. Stop using the URL when the Kaggle session ends and remove/replace the Vercel secrets. For reliable production availability, use a continuously running hosted GPU/API service instead.